In [1]:
class Document:
  def __init__(self, doc_id, content):
    self.doc_id = doc_id
    self.content = content
    self.shingles = set()

In [2]:
import pandas as pd

with open('/content/drive/MyDrive/IR_Lab/med/MED.ALL') as file:
  all_data = file.read()

raw_docs = all_data.split('.I ')
documents = []

for doc in raw_docs:
    if not doc.strip():
        continue
    # Separate the document ID from the text
    parts = doc.split('.W\n', 1)
    doc_id = int(parts[0].strip())
    # Replace newlines inside text with spaces
    text = parts[1].replace('\n', ' ').strip() if len(parts) > 1 else ""
    documents.append(Document(doc_id, text))

## Min Hashing

In [ ]:
def get_shingles(text, k): # N-grams
    words = text.lower().split()
    return set([" ".join(words[i:i+k]) for i in range(len(words)-k+1)])

for doc in documents:
  doc.shingles = get_shingles(doc.content, 2)

all_shingles = set()
for doc in documents:
  all_shingles.update(doc.shingles)

vocab = list(all_shingles)
shingles_to_index = {shingle: i for i, shingle in enumerate(all_shingles)}

print(f"Total Unique Shingles (Vocabulary Size): {len(vocab)}")

Total Unique Shingles (Vocabulary Size): 88328


In [8]:
import numpy as np
import random

# Generate Universal Hash Functions
# Hash formula: h(x) = (a * x + b) % c (where c is a prime > vocab_size)

num_hashes = 100  # No. of hashes
prime = 1000003  # Large prime number > vocab_size

# Generate random hash coefficients: h(x) = (a * x + b) % prime
random.seed(42)
a_coefficients = [random.randint(1, prime) for _ in range(num_hashes)]
b_coefficients = [random.randint(0, prime) for _ in range(num_hashes)]


doc_signatures = {}

for doc in documents:
  shingle_ids = [shingles_to_index[shingle] for shingle in doc.shingles]

  # Compute MinHash signature
  doc_signature = []
  for i in range(num_hashes):
      a = a_coefficients[i]
      b = b_coefficients[i]
      if shingle_ids:
          min_hash = min((a * s_id + b) % prime for s_id in shingle_ids)
      else:
          min_hash = float('inf')
      doc_signature.append(min_hash)

  doc_signatures[doc.doc_id] = doc_signature

In [9]:
def jaccard_similarity(doc1, doc2):
    intersection = doc1.shingles & doc2.shingles
    union = doc1.shingles | doc2.shingles
    if not union:
        return 0
    return len(intersection) / len(union)


def minhash_similarity(sig1, sig2):
    matches = sum(a == b for a, b in zip(sig1, sig2))
    return matches / len(sig1)


# Pick two documents
doc1 = documents[0]
doc2 = documents[1]

# Actual Jaccard
actual = jaccard_similarity(doc1, doc2)

# MinHash estimate
estimated = minhash_similarity(
    doc_signatures[doc1.doc_id],
    doc_signatures[doc2.doc_id]
)

print("Document 1:", doc1.doc_id)
print("Document 2:", doc2.doc_id)
print("Actual Jaccard:", actual)
print("MinHash Estimated Jaccard:", estimated)

Document 1: 1
Document 2: 2
Actual Jaccard: 0.02
MinHash Estimated Jaccard: 0.01


In [7]:
def get_shingles(text, k):
  words = text.lower().split()
  return set([" ".join(words[i:i+k]) for i in range(len(words)-k+1)])

for doc in documents:
  doc.shingles = get_shingles(doc.content, 2)

all_shingles = set()
for doc in documents:
  all_shingles.update(doc.shingles)

vocab = list(all_shingles)
shingles_to_index = {shingle: i for i, shingle in enumerate(all_shingles)}

print(f"Total Unique Shingles (Vocabulary Size): {len(vocab)}")


# h(x) = (a.x + b) % c
prime = 1000007
a_coefficients = [random.randint(1, prime) for _ in range(num_hashes)]
b_coefficients = [random.randint(0, prime) for _ in range(num_hashes)]

num_hashes = 10

doc_signatures = {}

for doc in documents:
  shingle_ids = [shingles_to_index[shingle] for shingle in doc.shingles]

  # Compute MinHash signature
  doc_signature = []
  for i in range(num_hashes):
      a = a_coefficients[i]
      b = b_coefficients[i]
      if shingle_ids:
          min_hash = min((a * s_id + b) % prime for s_id in shingle_ids)
      else:
          min_hash = float('inf')
      doc_signature.append(min_hash)

  doc_signatures[doc.doc_id] = doc_signature

Total Unique Shingles (Vocabulary Size): 88328


#Page rank

In [2]:
import numpy as np

A = np.array([
    [0, 1, 1, 0],
    [0, 0, 1, 1],
    [1, 0, 0, 0],
    [0, 0, 1, 0]
], dtype=float)

n = len(A)
d = 0.85

# Number of outgoing links from each page
out_degree = A.sum(axis=1)

# Transition matrix
M = A / out_degree[:, None]

# Initial PageRank
pr = np.ones(n) / n

for i in range(100):
    pr = (1-d)/n + d * M.T @ pr
    # print(f"Iteration {i+1}: {pr}")
print(pr)

[0.34261229 0.18311022 0.35895564 0.11532185]


In [14]:
import numpy as np

A = np.array([
    [0, 1, 0],
    [1, 0, 1],
    [0, 1, 0]
], dtype=float)

n = len(A)
a = 0.5

# S = a*(1/n) + (1-a).H

# Count outgoing links
row_sums = A.sum(axis=1)

# Create transition matrix
H = np.zeros_like(A)

for i in range(n):
    if row_sums[i] == 0:
        # Dangling node → distribute equally to all pages
        H[i] = 1 / n
    else:
        H[i] = A[i] / row_sums[i]

print("Transition matrix/Hyperlink matrix:")
print(H)

S = (1 - a)*H + (a)*(1/n)
print(S)

# Initial PageRank
pr = np.ones((n, 1)) / n

print("\n")

# -------------- Approx solution ---------------------
# for i in range(10):
#     pr = S @ pr
#     print(f"Iteration {i+1}: {pr.flatten()}")

max_iter = 100
tol = 1e-6
for _ in range(max_iter):
    pr_new = S @ pr
    if np.linalg.norm(pr_new - pr, ord=1) < tol:
        pr = pr_new
        break
    pr = pr_new
print(pr.flatten())

Transition matrix/Hyperlink matrix:
[[0.  1.  0. ]
 [0.5 0.  0.5]
 [0.  1.  0. ]]
[[0.16666667 0.66666667 0.16666667]
 [0.41666667 0.16666667 0.41666667]
 [0.16666667 0.66666667 0.16666667]]


[0.33333333 0.33333333 0.33333333]


In [19]:
import numpy as np

A = np.array([
    [0, 1, 0],
    [1, 0, 1],
    [0, 1, 0]
], dtype=float)

n = len(A)
a = 0.5

# ---------------- Hyperlink / Transition matrix ----------------

row_sums = A.sum(axis=1)

H = np.zeros_like(A)

for i in range(n):
    if row_sums[i] == 0:
        # Dangling node → distribute equally to all pages
        H[i] = 1 / n
    else:
        H[i] = A[i] / row_sums[i]

print("Transition matrix H:")
print(H)

# Since we use column PageRank: G = (1-a)H^T + a/n
G = (1 - a) * H.T + a * np.ones((n, n)) / n

print("\nGoogle matrix G:")
print(G)

# Initial PageRank
pr = np.ones(n) / n


# ---------------- Approx. solution ----------------

max_iter = 1000
tol = 1e-6
for _ in range(max_iter):
    pr_new = G @ pr
    if np.linalg.norm(pr_new - pr, ord=1) < tol:
        pr = pr_new
        break
    pr = pr_new

print("\nApproximate PageRank:")
print(pr)

# ---------------- Exact solution ----------------

# Solve: pr = G @ pr
# Also: sum(pr) = 1

B = np.eye(n) - G

# Replace one equation with normalization condition
B[-1] = np.ones(n)

b = np.zeros(n)
b[-1] = 1

pr_exact = np.linalg.solve(B, b)

print("\nExact PageRank:")
print(pr_exact)

Transition matrix H:
[[0.  1.  0. ]
 [0.5 0.  0.5]
 [0.  1.  0. ]]

Google matrix G:
[[0.16666667 0.41666667 0.16666667]
 [0.66666667 0.16666667 0.66666667]
 [0.16666667 0.41666667 0.16666667]]

Approximate PageRank:
[0.27777783 0.44444434 0.27777783]

Exact PageRank:
[0.27777778 0.44444444 0.27777778]


In [22]:
import numpy as np

A = np.array([
    [0, 1, 0, 1, 1],
    [1, 0, 1, 1, 1],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0]
], dtype=float)

n = len(A)
a = 0.5

# ---------------- Hyperlink / Transition matrix ----------------

row_sums = A.sum(axis=1)

H = np.zeros_like(A)

for i in range(n):
    if row_sums[i] == 0:
        # Dangling node → distribute equally to all pages
        H[i] = 1 / n
    else:
        H[i] = A[i] / row_sums[i]

print("Transition matrix H:")
print(H)

# Since we use column PageRank: G = (1-a)H^T + a/n
G = (1 - a)*H.T + a * np.ones((n, n)) / n

print("\nGoogle matrix G:")
print(G)


pr = np.ones(n) / n


# ---------------- Approx. solution ----------------

max_iter = 1000
# tol = 1e-6
for _ in range(max_iter):
    pr_new = G @ pr
    # if np.linalg.norm(pr_new - pr, ord=1) < tol:
    if np.allclose(pr_new, pr):
        pr = pr_new
        break
    pr = pr_new

print("\nApproximate PageRank:")
print(pr)

# ---------------- Exact solution ----------------

# Solve: pr = G @ pr
# Also: sum(pr) = 1

B = np.eye(n) - G

# Replace one equation with normalization condition
B[-1] = np.ones(n)

b = np.zeros(n)
b[-1] = 1

pr_exact = np.linalg.solve(B, b)

print("\nExact PageRank:")
print(pr_exact)

Transition matrix H:
[[0.         0.33333333 0.         0.33333333 0.33333333]
 [0.25       0.         0.25       0.25       0.25      ]
 [0.33333333 0.33333333 0.         0.33333333 0.        ]
 [0.2        0.2        0.2        0.2        0.2       ]
 [0.         0.5        0.5        0.         0.        ]]

Google matrix G:
[[0.1        0.225      0.26666667 0.2        0.1       ]
 [0.26666667 0.1        0.26666667 0.2        0.35      ]
 [0.1        0.225      0.1        0.2        0.35      ]
 [0.26666667 0.225      0.26666667 0.2        0.1       ]
 [0.26666667 0.225      0.1        0.2        0.1       ]]

Approximate PageRank:
[0.1824535  0.22929079 0.19503671 0.21286256 0.18035644]

Exact PageRank:
[0.18245369 0.22929046 0.1950367  0.21286264 0.18035652]


#HITS

In [23]:
import numpy as np

# Adjacency matrix
# A[i][j] = 1 means page i -> page j

A = np.array([
    [0, 1, 1],
    [0, 0, 1],
    [0, 0, 0]
], dtype=float)

n = len(A)

# Initial hub and authority scores as 1/n
hub = np.ones(n)/n
authority = np.ones(n)/n

# Number of iterations
iterations = 100

for i in range(iterations):
    # Authority = A^T × Hub
    new_authority = A.T @ hub
    new_authority = new_authority / np.sum(new_authority)

    # Hub = A × Authority
    new_hub = A @ authority
    new_hub = new_hub / np.sum(new_hub)

    # print(f"\nIteration {i+1}")
    # print("Authority:", new_authority)
    # print("Hub      :", new_hub)

    if np.allclose(new_authority, authority) and np.allclose(new_hub, hub):
        break

    hub = new_hub
    authority = new_authority

print("\nFinal scores:")
print("Authority:", authority)
print("Hub      :", hub)


Final scores:
Authority: [0.         0.38196721 0.61803279]
Hub      : [0.61803279 0.38196721 0.        ]


In [24]:
# Exact HITS using eigenvectors

# Hub matrix
HH = A @ A.T

# Authority matrix
AA = A.T @ A

# Eigenvalues and eigenvectors
eig_h, vec_h = np.linalg.eig(HH)
eig_a, vec_a = np.linalg.eig(AA)

# Principal eigenvectors
hub_exact = vec_h[:, np.argmax(eig_h)]
authority_exact = vec_a[:, np.argmax(eig_a)]

# Make positive
hub_exact = np.abs(hub_exact)
authority_exact = np.abs(authority_exact)

# Normalize so sum = 1
hub_exact = hub_exact / np.sum(hub_exact)
authority_exact = authority_exact / np.sum(authority_exact)

print("Exact Hub:", hub_exact)
print("Exact Authority:", authority_exact)

Exact Hub: [0.61803399 0.38196601 0.        ]
Exact Authority: [0.         0.38196601 0.61803399]


# SVD

In [28]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "The cat drinks milk",
    "The kitten drinks water",
    "The dog eats meat",
    "The puppy eats food",
    "Cats and kittens are pets",
]

vectorizer = CountVectorizer(stop_words="english")
td_matrix = vectorizer.fit_transform(documents)

A = td_matrix.toarray()
print(A)
m,n = A.shape

# calc A_T.A
ATA = A.T @ A

# calc eigen val and vec
eigenvalues, eigenvectors = np.linalg.eigh(ATA)
idx = np.argsort(eigenvalues)[::-1]

eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Singluar values
singular_values = np.sqrt(np.maximum(eigenvalues, 0))


# A = U.sigma.V_T

V = eigenvectors

# u_i = A.v_i / sig_i
U = np.zeros((m,m))
for i in range(min(m,n)):
    if singular_values[i] > 1e-10:
        U[:,i] = (A @ V[:,i]) / singular_values[i]

Sigma = np.zeros((m, n))
for i in range(min(m, n)):
    Sigma[i, i] = singular_values[i]


A_reconstructed = U @ Sigma @ V.T
print("\nReconstructed A:")
print(A_reconstructed)


[[1 0 0 1 0 0 0 0 0 1 0 0 0]
 [0 0 0 1 0 0 1 0 0 0 0 0 1]
 [0 0 1 0 1 0 0 0 1 0 0 0 0]
 [0 0 0 0 1 1 0 0 0 0 0 1 0]
 [0 1 0 0 0 0 0 1 0 0 1 0 0]]

Reconstructed A:
[[ 1.00000000e+00 -3.04932623e-16  6.11943216e-18  1.00000000e+00
   6.54008894e-18  4.20656771e-19 -5.55111512e-17 -1.15009313e-16
   6.11943216e-18  1.00000000e+00 -1.15009313e-16  4.20656771e-19
  -5.55111512e-17]
 [ 5.55111512e-17 -1.24931430e-16  2.17558741e-17  1.00000000e+00
  -6.26850286e-18 -2.80243769e-17  1.00000000e+00 -6.66643754e-17
   2.17558741e-17 -2.85285652e-34 -6.66643754e-17 -2.80243769e-17
   1.00000000e+00]
 [ 5.53147652e-17  8.16209300e-17  1.00000000e+00  2.09844055e-17
   1.00000000e+00 -2.22044605e-16 -7.85046229e-17  9.07891826e-17
   1.00000000e+00 -5.75202174e-17  6.30336070e-17 -2.77555756e-16
   9.47634627e-17]
 [ 2.09936135e-17 -5.24663554e-17  2.22044605e-16  3.24462060e-17
   1.00000000e+00  1.00000000e+00 -7.85046229e-17  4.64989940e-17
   2.22044605e-16 -4.60584170e-17  7.42545696e-17  1.

In [25]:
import numpy as np

# Example matrix
A = np.array([
    [3, 0],
    [0, 2]
], dtype=float)

m, n = A.shape

# -------------------------------------------------
# Step 1: Calculate A^T A
# -------------------------------------------------

ATA = A.T @ A

print("A^T A:")
print(ATA)


# -------------------------------------------------
# Step 2: Find eigenvalues and eigenvectors
# -------------------------------------------------

eigenvalues, eigenvectors = np.linalg.eig(ATA)

# Sort in descending order
idx = np.argsort(eigenvalues)[::-1]

eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("\nEigenvalues:")
print(eigenvalues)

print("\nEigenvectors:")
print(eigenvectors)


# -------------------------------------------------
# Step 3: Singular values
# σ = sqrt(λ)
# -------------------------------------------------

singular_values = np.sqrt(np.maximum(eigenvalues, 0))

print("\nSingular values:")
print(singular_values)


# -------------------------------------------------
# Step 4: V
# -------------------------------------------------

V = eigenvectors

print("\nV:")
print(V)


# -------------------------------------------------
# Step 5: Calculate U
# u_i = A v_i / σ_i
# -------------------------------------------------

U = np.zeros((m, m))

for i in range(min(m, n)):

    if singular_values[i] > 1e-10:
        U[:, i] = (A @ V[:, i]) / singular_values[i]

print("\nU:")
print(U)


# -------------------------------------------------
# Step 6: Sigma matrix
# -------------------------------------------------

Sigma = np.zeros((m, n))

for i in range(min(m, n)):
    Sigma[i, i] = singular_values[i]

print("\nSigma:")
print(Sigma)


# -------------------------------------------------
# Step 7: Reconstruct A
# -------------------------------------------------

A_reconstructed = U @ Sigma @ V.T

print("\nReconstructed A:")
print(A_reconstructed)

A^T A:
[[9. 0.]
 [0. 4.]]

Eigenvalues:
[9. 4.]

Eigenvectors:
[[1. 0.]
 [0. 1.]]

Singular values:
[3. 2.]

V:
[[1. 0.]
 [0. 1.]]

U:
[[1. 0.]
 [0. 1.]]

Sigma:
[[3. 0.]
 [0. 2.]]

Reconstructed A:
[[3. 0.]
 [0. 2.]]


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# 1. Sample document collection
# -----------------------------

documents = [
    "The cat drinks milk",
    "The kitten drinks water",
    "The dog eats meat",
    "The puppy eats food",
    "Cats and kittens are pets",
]

# -----------------------------
# 2. Convert documents into
#    a TF-IDF term-document matrix
# -----------------------------

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(documents)

print("Original matrix shape:", X.shape)
print(X)
print("Terms:", vectorizer.get_feature_names_out())

# -----------------------------
# 3. Apply SVD
# -----------------------------

svd = TruncatedSVD(n_components=2, random_state=42)

X_reduced = svd.fit_transform(X)

print("\nReduced document representation:")
print(X_reduced)

# -----------------------------
# 4. Convert a query into the
#    same semantic space
# -----------------------------

query = ["kitten drinking milk"]

query_vector = vectorizer.transform(query)

query_reduced = svd.transform(query_vector)

# -----------------------------
# 5. Calculate similarity
#    between query and documents
# -----------------------------

similarities = cosine_similarity(query_reduced, X_reduced)

# -----------------------------
# 6. Rank documents
# -----------------------------

ranking = np.argsort(similarities[0])[::-1]

print("\nSearch results:")

for i in ranking:
    print(
        f"Score: {similarities[0][i]:.3f} | "
        f"Document: {documents[i]}"
    )

Original matrix shape: (5, 13)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 15 stored elements and shape (5, 13)>
  Coords	Values
  (0, 0)	0.6141889663426562
  (0, 3)	0.49552379079705033
  (0, 9)	0.6141889663426562
  (1, 3)	0.49552379079705033
  (1, 6)	0.6141889663426562
  (1, 12)	0.6141889663426562
  (2, 2)	0.6141889663426562
  (2, 4)	0.49552379079705033
  (2, 8)	0.6141889663426562
  (3, 4)	0.49552379079705033
  (3, 11)	0.6141889663426562
  (3, 5)	0.6141889663426562
  (4, 1)	0.5773502691896258
  (4, 7)	0.5773502691896258
  (4, 10)	0.5773502691896258
Terms: ['cat' 'cats' 'dog' 'drinks' 'eats' 'food' 'kitten' 'kittens' 'meat'
 'milk' 'pets' 'puppy' 'water']

Reduced document representation:
[[-3.16800034e-18  7.89158991e-01]
 [-9.76091774e-16  7.89158991e-01]
 [ 7.89158991e-01  3.80144522e-16]
 [ 7.89158991e-01  3.91185277e-16]
 [-2.77916334e-15 -2.67589568e-16]]

Search results:
Score: 1.000 | Document: The cat drinks milk
Score: 1.000 | Document: The kitten drinks wat

In [27]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "The cat drinks milk",
    "The kitten drinks water",
    "The dog eats meat",
    "The puppy eats food",
    "Cats and kittens are pets",
]

vectorizer = CountVectorizer(stop_words="english")
td_matrix = vectorizer.fit_transform(documents)

print(td_matrix.toarray())

[[1 0 0 1 0 0 0 0 0 1 0 0 0]
 [0 0 0 1 0 0 1 0 0 0 0 0 1]
 [0 0 1 0 1 0 0 0 1 0 0 0 0]
 [0 0 0 0 1 1 0 0 0 0 0 1 0]
 [0 1 0 0 0 0 0 1 0 0 1 0 0]]


In [ ]:
import numpy as np

# Example matrix
A = np.array([
    [3, 0],
    [0, 2]
], dtype=float)

m, n = A.shape

# -------------------------------------------------
# Step 1: Calculate A^T A
# -------------------------------------------------

ATA = A.T @ A

print("A^T A:")
print(ATA)


# -------------------------------------------------
# Step 2: Find eigenvalues and eigenvectors
# -------------------------------------------------

eigenvalues, eigenvectors = np.linalg.eig(ATA)

# Sort in descending order
idx = np.argsort(eigenvalues)[::-1]

eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("\nEigenvalues:")
print(eigenvalues)

print("\nEigenvectors:")
print(eigenvectors)


# -------------------------------------------------
# Step 3: Singular values
# σ = sqrt(λ)
# -------------------------------------------------

singular_values = np.sqrt(np.maximum(eigenvalues, 0))

print("\nSingular values:")
print(singular_values)


# -------------------------------------------------
# Step 4: V
# -------------------------------------------------

V = eigenvectors

print("\nV:")
print(V)


# -------------------------------------------------
# Step 5: Calculate U
# u_i = A v_i / σ_i
# -------------------------------------------------

U = np.zeros((m, m))

for i in range(min(m, n)):

    if singular_values[i] > 1e-10:
        U[:, i] = (A @ V[:, i]) / singular_values[i]

print("\nU:")
print(U)


# -------------------------------------------------
# Step 6: Sigma matrix
# -------------------------------------------------

Sigma = np.zeros((m, n))

for i in range(min(m, n)):
    Sigma[i, i] = singular_values[i]

print("\nSigma:")
print(Sigma)


# -------------------------------------------------
# Step 7: Reconstruct A
# -------------------------------------------------

A_reconstructed = U @ Sigma @ V.T

print("\nReconstructed A:")
print(A_reconstructed)